# Tensorflow2.0 小练习

In [1]:
import tensorflow as tf
import numpy as np

## 实现softmax函数

In [2]:
def softmax(x):
    # 将输入转为 float32 张量
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    # 减去最大值防止指数爆炸 (数值稳定性优化)
    x_max = tf.reduce_max(x, axis=-1, keepdims=True)
    exp_x = tf.exp(x - x_max)
    # 计算概率
    prob_x = exp_x / tf.reduce_sum(exp_x, axis=-1, keepdims=True)
    return prob_x

test_data = np.random.normal(size=[10, 5])
(softmax(test_data).numpy() - tf.nn.softmax(test_data, axis=-1).numpy())**2 <0.0001

array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现sigmoid函数

In [3]:
def sigmoid(x):
    # 将输入转为 float32 张量
    x = tf.convert_to_tensor(x, dtype=tf.float32)
    # 直接套用公式
    prob_x = 1.0 / (1.0 + tf.exp(-x))
    return prob_x

test_data = np.random.normal(size=[10, 5])
(sigmoid(test_data).numpy() - tf.nn.sigmoid(test_data).numpy())**2 < 0.0001

array([[ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True],
       [ True,  True,  True,  True,  True]])

## 实现 softmax 交叉熵loss函数

In [4]:
def softmax_ce(x, label):
    x = tf.convert_to_tensor(x)
    # 将 label 强制转换为与 x 相同的类型
    label = tf.cast(label, dtype=x.dtype)
    
    # 保证截断的边界值也和 x 的类型一致，防止报错
    clip_min = tf.cast(1e-12, x.dtype)
    clip_max = tf.cast(1.0, x.dtype)
    x_safe = tf.clip_by_value(x, clip_min, clip_max)
    
    sample_loss = -tf.reduce_sum(label * tf.math.log(x_safe), axis=-1)
    loss = tf.reduce_mean(sample_loss)
    return loss

test_data = np.random.normal(size=[10, 5])
prob = tf.nn.softmax(test_data)
label = np.zeros_like(test_data)
label[np.arange(10), np.random.randint(0, 5, size=10)]=1.

((tf.reduce_mean(tf.nn.softmax_cross_entropy_with_logits(label, test_data))
    - softmax_ce(prob, label))**2 < 0.0001).numpy()

np.True_

## 实现 sigmoid 交叉熵loss函数

In [5]:
def sigmoid_ce(x, label):
    x = tf.convert_to_tensor(x)
    label = tf.cast(label, dtype=x.dtype)
    
    clip_min = tf.cast(1e-12, x.dtype)
    clip_max = tf.cast(1.0 - 1e-12, x.dtype)
    x_safe = tf.clip_by_value(x, clip_min, clip_max)
    
    # 公式中的 1.0 也要确保类型一致
    one = tf.cast(1.0, x.dtype)
    sample_loss = -(label * tf.math.log(x_safe) + (one - label) * tf.math.log(one - x_safe))
    
    loss = tf.reduce_mean(sample_loss)
    return loss

test_data = np.random.normal(size=[10])
prob = tf.nn.sigmoid(test_data)
label = np.random.randint(0, 2, 10).astype(test_data.dtype)
print (label)

((tf.reduce_mean(tf.nn.sigmoid_cross_entropy_with_logits(label, test_data))- sigmoid_ce(prob, label))**2 < 0.0001).numpy()


[0. 1. 1. 1. 1. 0. 1. 0. 1. 1.]


np.True_